# S3 Semantics Locally: URIs, Prefixes, a boto3-like Client, Event-Driven Processing

## What you'll practice

S3 isn't a filesystem: it's a flat key→object store where "folders" are just key prefixes, listing is paginated, there's no rename, and writes are whole-object.
`datasets/s3/` holds three buckets (`lab-raw-bucket`, `lab-processed-bucket`, `lab-config-bucket`). You'll implement S3-like operations over it with the **same names and
response shapes as boto3**, so moving to real AWS later is a one-line change (`boto3.client("s3")`).

> Function and class names in **Implement** are what the tests look for. Run all tests for this notebook with `pytest tests/aws/test_11_01_s3_simulation.py -v` from the lab root. Untouched exercises show as *skipped*.
>
> Solutions: `solutions/11_python_for_aws/01_s3_simulation.ipynb`. Don't open it until you've tried.

### Setup — run this first (sample data and imports, not a solution)

In [ ]:
from pathlib import Path
LAB = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datasets").is_dir() and (p / "labkit").is_dir())   # lab root (works from any folder)
S3_ROOT = LAB / "datasets/s3"        # local stand-in for S3: <root>/<bucket>/<key>
for p in sorted(S3_ROOT.rglob("*"))[:12]:
    print(p.relative_to(S3_ROOT))

## Exercise 1 — 🟢 Easy · Map S3 URIs to local paths safely

**Implement:** `s3_to_local(uri: str, root) -> Path`

`s3://bucket/a/b.csv` → `root/bucket/a/b.csv`. Reject non-`s3://` URIs, empty buckets, and any key containing a `..` path segment (path traversal) with `ValueError`. A trailing `/` (a 'folder') maps to the directory.

**Input:** `uri: str, root`

**Expected output:** `Path`

**Constraints:** Standard library only, unless the statement says otherwise.

**Example:** the tests are executable examples — read `test_s3_to_local` in `tests/aws/test_11_01_s3_simulation.py`.

**Check:** `pytest tests/aws/test_11_01_s3_simulation.py -k test_s3_to_local`

**YOUR CODE ↓**

## Exercise 2 — 🟡 Medium · List with prefix and delimiter

**Implement:** `list_prefix(root, bucket: str, prefix: str = '', delimiter: str | None = None) -> dict`

Emulate `ListObjectsV2`: return `{"Contents": [{"Key", "Size"}...], "CommonPrefixes": [{"Prefix"}...]}` for objects whose key starts with `prefix` (keys use `/`, sorted). With
`delimiter="/"`, keys that have another `/` after the prefix are rolled up into unique `CommonPrefixes` (`"sales/ingest_date=2026-01-10/"`) instead of being listed. Hidden files (`.keep`) count as objects.
Missing bucket → `FileNotFoundError`.

**Input:** `root, bucket: str, prefix: str = '', delimiter: str | None = None`

**Expected output:** `dict`

**Constraints:** Standard library only, unless the statement says otherwise.

**Example:** the tests are executable examples — read `test_list_prefix` in `tests/aws/test_11_01_s3_simulation.py`.

**Check:** `pytest tests/aws/test_11_01_s3_simulation.py -k test_list_prefix`

**YOUR CODE ↓**

## Exercise 3 — 🟡 Medium · 'Rename' a prefix (copy + delete)

**Implement:** `move_prefix(root, src_uri: str, dst_uri: str, delete_source: bool = True) -> int`

S3 has no rename: moving a 'folder' means copying every object then deleting the originals. Copy all objects under `src_uri` (a prefix ending in `/`) to the same relative keys under
`dst_uri`. Only after **all** copies succeed, delete sources (if requested) and remove now-empty local directories. Return the number of objects. Refuse (`ValueError`) if destination is inside source.

**Input:** `root, src_uri: str, dst_uri: str, delete_source: bool = True`

**Expected output:** `int`

**Constraints:** Standard library only, unless the statement says otherwise.

**Example:** the tests are executable examples — read `test_move_prefix` in `tests/aws/test_11_01_s3_simulation.py`.

**Check:** `pytest tests/aws/test_11_01_s3_simulation.py -k test_move_prefix`

**YOUR CODE ↓**

## Exercise 4 — 🟠 Hard · A boto3-compatible local S3 client

**Implement:** `class ClientError(Exception)`, `class LocalS3Client`

Implement a subset of the boto3 S3 client over a local root, using boto3's **exact** method names, keyword arguments and response shapes:
`create_bucket(Bucket)`, `put_object(Bucket, Key, Body)` (bytes or str; returns `{"ETag": '"<md5 hex>"'}`), `get_object(Bucket, Key)` → `{"Body": file-like with .read(), "ContentLength", "ETag"}`,
`head_object(Bucket, Key)` → same without Body, `delete_object(Bucket, Key)` (idempotent; `{}`),
`list_objects_v2(Bucket, Prefix="", MaxKeys=1000, ContinuationToken=None)` → `{"Contents": [{"Key", "Size", "ETag"}], "KeyCount", "IsTruncated", "NextContinuationToken"?}` (paginated, sorted; omit
`Contents` when empty, like AWS). Errors raise `ClientError` with `err.response["Error"]["Code"]` in `NoSuchBucket` / `NoSuchKey` — so your error handling code is boto3-ready.

**Input:** see the class description above

**Expected output:** an instance behaving as described

**Constraints:** Standard library only, unless the statement says otherwise.

**Example:** the tests are executable examples — read `test_local_s3_client_crud, test_local_s3_client_pagination` in `tests/aws/test_11_01_s3_simulation.py`.

**Check:** `pytest tests/aws/test_11_01_s3_simulation.py -k "test_local_s3_client_crud or test_local_s3_client_pagination"`

**YOUR CODE ↓**

## Challenge — 🔴 Very Hard · An S3-event-driven, idempotent handler (Lambda style) · 🧩 Mini-project

**Implement:** `handle_s3_event(event: dict, root) -> dict`

Write a Lambda-style handler for S3 `ObjectCreated` notifications. `event["Records"][i]` has `eventName`, `s3.bucket.name`, `s3.object.key` (**URL-encoded**, spaces as `+` →
use `urllib.parse.unquote_plus`). For each `.csv` object created:
1. read it from the local S3 root; validate rows (quantity int ≥ 1, unit_price float ≥ 0, status in PLACED/SHIPPED/DELIVERED/CANCELLED/RETURNED case-insensitive, order_id present);
2. write valid rows as JSONL to `lab-processed-bucket/clean/<original key with .csv→.jsonl>`, rejects (with `_error`) to `lab-processed-bucket/rejected/<same>` (only if any);
3. be **idempotent**: compute the object's MD5; if marker `lab-processed-bucket/_markers/<bucket>/<key>.<md5>.done` exists, skip; write the marker last.
Ignore non-`ObjectCreated` events and non-CSV keys (count them as `ignored`). A missing object is recorded under `errors` (don't crash the whole batch).
Return `{"processed": [keys], "skipped": [keys], "ignored": int, "errors": [{"key", "error"}], "valid_rows": int, "rejected_rows": int}`.

**Input:** `event: dict, root`

**Expected output:** `dict`

**Constraints:** Standard library only, unless the statement says otherwise.

**Example:** the tests are executable examples — read `test_handle_s3_event` in `tests/aws/test_11_01_s3_simulation.py`.

**Check:** `pytest tests/aws/test_11_01_s3_simulation.py -k test_handle_s3_event`

**YOUR CODE ↓**